In [2]:
import json
import pandas as pd

with open("../data/raw/water/2026-11-24/osm-france-drinking-water.geojson", encoding="utf-8") as f:
    data = json.load(f)

print(data["type"], "-", len(data["features"]), "points")

df = pd.json_normalize(data["features"])
df["longitude"] = df["geometry.coordinates"].str[0]
df["latitude"] = df["geometry.coordinates"].str[1]

print(df["geometry.type"].value_counts())
print(df.columns.tolist())
df.head()

FeatureCollection - 36279 points
geometry.type
Point    36279
Name: count, dtype: int64
['type', 'geometry.type', 'geometry.coordinates', 'properties.name', 'properties.operator', 'properties.accepts_bottle', 'properties.fee', 'properties.indoor', 'properties.offers_cold_water', 'properties.offers_warm_water', 'properties.offers_hot_water', 'properties.description', 'properties.meta_name_com', 'properties.meta_code_com', 'properties.meta_name_dep', 'properties.meta_code_dep', 'properties.meta_name_reg', 'properties.meta_code_reg', 'properties.meta_osm_id', 'properties.meta_osm_type', 'properties.meta_osm_url', 'properties.meta_first_update', 'properties.meta_last_update', 'properties.meta_versions_number', 'properties.meta_users_number', 'longitude', 'latitude']


,type,geometry.type,geometry.coordinates,properties.name,properties.operator,properties.accepts_bottle,properties.fee,properties.indoor,properties.offers_cold_water,properties.offers_warm_water,...,properties.meta_code_reg,properties.meta_osm_id,properties.meta_osm_type,properties.meta_osm_url,properties.meta_first_update,properties.meta_last_update,properties.meta_versions_number,properties.meta_users_number,longitude,latitude
0,Feature,Point,"[6.571511, 45.505319]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,84,1428745388,node,https://www.openstreetmap.org/node/1428745388,2011-09-10T10:18:11+00:00,2011-09-10,1.0,1.0,6.571511,45.505319
1,Feature,Point,"[7.121554, 43.697659]",NaN,NaN,NaN,NaN,no,NaN,NaN,...,93,1434936674,node,https://www.openstreetmap.org/node/1434936674,2020-08-05T15:45:03+00:00,2025-12-27,2.0,2.0,7.121554,43.697659
2,Feature,Point,"[5.412537, 45.17049]",NaN,NaN,NaN,no,NaN,NaN,NaN,...,84,1448541017,node,https://www.openstreetmap.org/node/1448541017,2011-09-28T23:38:52+00:00,2013-08-23,2.0,2.0,5.412537,45.170490
3,Feature,Point,"[6.35506, 47.337771]",Le Domaine d'Aucroix,NaN,NaN,yes,NaN,NaN,NaN,...,27,26316560,way,https://www.openstreetmap.org/way/26316560,2008-08-15T22:24:24+00:00,2026-10-04,25.0,10.0,6.355060,47.337771
4,Feature,Point,"[1.707653, 45.660803]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,75,32480366,way,https://www.openstreetmap.org/way/32480366,2022-10-08T11:54:06+00:00,2026-10-10,5.0,3.0,1.707653,45.660803


In [3]:
print(len(df), "points")
print(df.columns.tolist())

# Taux de remplissage de chaque colonne (part des valeurs non vides)
print(df.notna().mean().sort_values().round(2))

# Doublons sur l'identifiant OSM
print("doublons :", df.duplicated(["properties.meta_osm_type", "properties.meta_osm_id"]).sum())

# Coordonnées
print("sans coordonnées :", df["latitude"].isna().sum())
print(df[["latitude", "longitude"]].describe())

# Répartition par région
print(df["properties.meta_code_reg"].value_counts())

36279 points
['type', 'geometry.type', 'geometry.coordinates', 'properties.name', 'properties.operator', 'properties.accepts_bottle', 'properties.fee', 'properties.indoor', 'properties.offers_cold_water', 'properties.offers_warm_water', 'properties.offers_hot_water', 'properties.description', 'properties.meta_name_com', 'properties.meta_code_com', 'properties.meta_name_dep', 'properties.meta_code_dep', 'properties.meta_name_reg', 'properties.meta_code_reg', 'properties.meta_osm_id', 'properties.meta_osm_type', 'properties.meta_osm_url', 'properties.meta_first_update', 'properties.meta_last_update', 'properties.meta_versions_number', 'properties.meta_users_number', 'longitude', 'latitude']
properties.offers_warm_water       0.01
properties.offers_cold_water       0.01
properties.offers_hot_water        0.01
properties.description             0.03
properties.indoor                  0.04
properties.operator                0.06
properties.accepts_bottle          0.10
properties.name       

In [4]:
print("doublons :", df.duplicated(["properties.meta_osm_type", "properties.meta_osm_id"]).sum())
print(df[["latitude", "longitude"]].describe().round(3))

# Les 15 départements avec le plus de points
print(df["properties.meta_name_dep"].value_counts().head(15))

# Candidat zone d'étude : le Rhône
rhone = df[df["properties.meta_code_dep"] == "69"]
print("Rhône :", len(rhone), "points")
print(rhone["properties.meta_name_com"].value_counts().head(10))

doublons : 0
        latitude  longitude
count  36279.000  36279.000
mean      45.093      3.736
std        6.705      6.182
min      -21.387    -61.782
25%       43.953      1.528
50%       45.364      3.623
75%       47.476      5.868
max       51.079     55.821
properties.meta_name_dep
Isère                      2123
Savoie                     1460
Paris                      1453
Haute-Savoie               1136
Alpes-Maritimes            1125
Rhône                      1052
Hautes-Alpes                903
Pyrénées-Atlantiques        890
Hérault                     857
Haute-Garonne               817
Gironde                     807
Drôme                       717
Bouches-du-Rhône            688
Alpes-de-Haute-Provence     658
Aveyron                     642
Name: count, dtype: int64
Rhône : 1052 points
properties.meta_name_com
Lyon 3e Arrondissement     70
Lyon 7e Arrondissement     61
Villeurbanne               58
Lyon 8e Arrondissement     52
Lyon 5e Arrondissement     46
Lyon 2e A